# Campus gym crowd: exploratory data analysis

This notebook uses the shared loader and schema checks, preserves all source rows, and saves figures under `reports/figures/` at 200 dpi. It is descriptive analysis; no model split or selection is done here. Calendar summaries use source-local time, while the shared parser sorts actual instants.

## Load data


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from src.config import *
from src.data import load_raw, validate_schema, parse_datetime
raw = load_raw()
validate_schema(raw)
df = parse_datetime(raw)
df["local_datetime"] = pd.to_datetime(df[DATE_COLUMN].astype("string").str[:SOURCE_LOCAL_DATETIME_LENGTH])
df["local_date"] = df.local_datetime.dt.normalize()
df["day_name"] = df.day_of_week.map(dict(enumerate(EDA_DAY_NAMES)))
df["weekend_label"] = df.is_weekend.map(EDA_WEEKEND_LABELS)
df["season"] = df.month.map(EDA_MONTH_TO_SEASON)
df["week_start"] = df.local_date - pd.to_timedelta(df.day_of_week, unit="D")
for p in EDA_FIGURE_PATHS.values(): p.parent.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", palette=EDA_COLOR_PALETTE, font_scale=EDA_FONT_SCALE)
plt.rcParams["figure.dpi"] = EDA_FIGURE_DPI
def save(fig, key):
    fig.tight_layout(); fig.savefig(EDA_FIGURE_PATHS[key], dpi=EDA_FIGURE_DPI, bbox_inches="tight"); plt.show()
print(f"{len(df):,} rows, {len(raw.columns)} raw columns; local date range {df.local_datetime.min()} to {df.local_datetime.max()}")


## 1. Target distribution

The histogram and boxplot show a right-tailed distribution, with positive skew (0.565). There are 8,775 zero readings (14.1%) and 355 readings above the upper 1.5-IQR fence. We keep zero and high-count readings because they may reflect real gym conditions; they are not removed automatically.

In [ ]:
target=df[TARGET_COLUMN]; q1,q3=target.quantile([.25,.75]); upper=q3+TARGET_OUTLIER_IQR_MULTIPLIER*(q3-q1); skew=target.skew(); zeros=int((target==ZERO_OCCUPANCY_VALUE).sum()); outliers=int((target>upper).sum())
fig,ax=plt.subplots(2,1,figsize=EDA_DISTRIBUTION_FIGURE_SIZE,gridspec_kw={"height_ratios":[3,1]}); sns.histplot(target,bins=EDA_HISTOGRAM_BINS,ax=ax[0]); ax[0].set(title="Gym occupancy distribution",xlabel="People in gym (count)",ylabel="Readings (count)"); sns.boxplot(x=target,ax=ax[1]); ax[1].set(xlabel="People in gym (count)"); save(fig,"target_distribution")
print(f"skew={skew:.3f}; zeros={zeros:,} ({100*zeros/len(df):.1f}%); above upper IQR fence={outliers:,}")

## 2. Full time series

Daily means make long-term changes easier to see. Resampling retains missing dates as gaps, rather than filling or interpolating them.

In [ ]:
daily=df.set_index("local_datetime")[TARGET_COLUMN].resample(EDA_DAILY_FREQUENCY).mean(); fig,ax=plt.subplots(figsize=EDA_WIDE_FIGURE_SIZE); ax.plot(daily.index,daily,linewidth=EDA_LINE_WIDTH); ax.set(title="Daily mean occupancy across the full dataset",xlabel="Date (source local time)",ylabel="Mean people (count)"); save(fig,"full_time_series")

## 3. Representative complete week

This is the complete Monday-to-Sunday week whose mean is closest to the median of complete weeks. Each day rises from overnight zeros to daytime activity, while the weekend peaks are lower than the early-week peaks. The selection rule is reproducible, but one example does not represent every week.

In [ ]:
weeks=df.groupby("week_start")[TARGET_COLUMN].agg(["mean","size"]); complete=weeks[weeks["size"]>=EDA_READINGS_PER_DAY*EDA_WEEK_DAYS]; week_start=(complete["mean"]-complete["mean"].median()).abs().idxmin(); w=df[df.week_start==week_start]; fig,ax=plt.subplots(figsize=EDA_WIDE_FIGURE_SIZE); ax.plot(w.local_datetime,w[TARGET_COLUMN],linewidth=EDA_LINE_WIDTH); ax.set(title=f"Representative week from {week_start:%Y-%m-%d}",xlabel="Date/time (source local)",ylabel="People (count)"); save(fig,"representative_week"); print(f"week mean={w[TARGET_COLUMN].mean():.2f}")

## 4. Representative day

This date has a daily mean closest to the median daily mean. Counts fall to zero overnight, rise after about 06:00, and are highest in the evening. The within-day curve shows variation hidden by a daily average.

In [ ]:
ds=df.groupby("local_date")[TARGET_COLUMN].agg(["mean","size"]); date=(ds["mean"]-ds["mean"].median()).abs().idxmin(); d=df[df.local_date==date]; fig,ax=plt.subplots(figsize=EDA_FIGURE_SIZE); ax.plot(d.local_datetime,d[TARGET_COLUMN],marker="o",markersize=2); ax.set(title=f"Representative day: {date:%A %Y-%m-%d}",xlabel="Time (source local)",ylabel="People (count)"); ax.tick_params(axis="x",rotation=EDA_X_LABEL_ROTATION); save(fig,"representative_day")

## 5. Hourly weekday/weekend profile

Both profiles have a daily rhythm, with different peak timing and levels. This supports cyclic hour encoding and keeping a weekend indicator; hour is not a linear quantity.

In [ ]:
h=df.groupby(["weekend_label","hour"],observed=True)[TARGET_COLUMN].mean().reset_index(); fig,ax=plt.subplots(figsize=EDA_FIGURE_SIZE); sns.lineplot(data=h,x="hour",y=TARGET_COLUMN,hue="weekend_label",marker="o",ax=ax); ax.set(title="Mean occupancy by hour and calendar type",xlabel="Hour of day (0?23)",ylabel="Mean people (count)"); ax.legend(title="Day type"); save(fig,"hourly_weekday_weekend")

## 6. Day-by-hour heatmap

The heatmap reveals that crowd levels depend on both weekday and hour. The busiest cell is Tuesday at 18:00 (54.72 people); this motivates representing the day and hour jointly.

In [ ]:
heat=df.pivot_table(index="day_name",columns="hour",values=TARGET_COLUMN,aggfunc="mean").reindex(EDA_DAY_NAMES); stacked=heat.stack(); max_day,max_hour=stacked.idxmax(); max_value=stacked.max(); fig,ax=plt.subplots(figsize=EDA_WIDE_FIGURE_SIZE); sns.heatmap(heat,cmap=EDA_COLOR_PALETTE,ax=ax,cbar_kws={"label":"Mean people (count)"}); ax.set(title="Mean occupancy by weekday and hour",xlabel="Hour (0?23)",ylabel="Day of week"); save(fig,"day_hour_heatmap"); print(f"maximum: {max_day} {max_hour}:00 = {max_value:.2f}")

## 7. Semester and holiday flags

Semester flags show distribution differences, while holidays have sparse support. These are associations, not causal effects; holiday comparisons especially need caution because few observations carry that flag.

In [ ]:
fig,axes=plt.subplots(1,len(EDA_FLAG_COLUMNS),figsize=EDA_FLAG_FIGURE_SIZE)
for ax,(col,label) in zip(axes,EDA_FLAG_COLUMNS.items()):
    t=df.assign(flag=df[col].map(EDA_BINARY_LABELS)); sns.boxplot(data=t,x="flag",y=TARGET_COLUMN,order=["No","Yes"],showfliers=False,ax=ax); ax.set(title=label,xlabel="Flag",ylabel="People (count)")
    print(col,df.groupby(col)[TARGET_COLUMN].agg(["mean","size"]).to_dict("index"))
fig.suptitle("Occupancy by semester and holiday flags"); save(fig,"calendar_flags")

## 8. Temperature versus occupancy

A positive trend is visible, but the wide scatter means temperature alone does not determine crowd size. The straight trend summarizes association, not causation.

In [ ]:
r=df[["temperature",TARGET_COLUMN]].corr().iloc[0,1]; fig,ax=plt.subplots(figsize=EDA_FIGURE_SIZE); sns.regplot(data=df,x="temperature",y=TARGET_COLUMN,scatter_kws={"alpha":EDA_SCATTER_ALPHA,"s":EDA_SCATTER_SIZE},ax=ax); ax.set(title=f"Temperature and occupancy (Pearson r={r:.2f})",xlabel="Temperature (?F)",ylabel="People (count)"); save(fig,"temperature_scatter"); print(f"Pearson r={r:.3f}")

## 9. Temperature by season

Seasonal temperature distributions overlap, and this file contains fewer than two full years. Seasonal differences are therefore descriptive and should not be treated as stable climatology.

In [ ]:
fig,ax=plt.subplots(figsize=EDA_FIGURE_SIZE); sns.boxplot(data=df,x="season",y="temperature",order=EDA_SEASONS,showfliers=False,ax=ax); ax.set(title="Temperature by meteorological season",xlabel="Season",ylabel="Temperature (?F)"); save(fig,"temperature_season")

## 10. Monthly trend and seasonality

Chronological monthly means show changes between particular months; the second panel pools the same month across years. Large chronological variation and limited years mean month alone will not capture all shifts.

In [ ]:
monthly=df.set_index("local_datetime")[TARGET_COLUMN].resample("MS").mean(); profile=df.groupby("month")[TARGET_COLUMN].mean().reindex(range(1,13)); fig,ax=plt.subplots(1,2,figsize=EDA_WIDE_FIGURE_SIZE); ax[0].plot(monthly.index,monthly,marker="o"); ax[0].set(title="Monthly means in time order",xlabel="Month",ylabel="Mean people (count)"); ax[1].plot(profile.index,profile,marker="o"); ax[1].set(title="Average by calendar month",xlabel="Month (1?12)",ylabel="Mean people (count)",xticks=range(1,13)); save(fig,"monthly_trend")

## 11. Numeric correlation

Correlation only captures pairwise linear relationships; it does not represent the non-linear cyclical hour effect. Treat this as a descriptive screening chart, not an automatic feature-selection rule.

In [ ]:
numeric=df.select_dtypes(include="number").drop(columns=["timestamp"],errors="ignore"); fig,ax=plt.subplots(figsize=EDA_WIDE_FIGURE_SIZE); sns.heatmap(numeric.corr(),cmap=EDA_COLOR_PALETTE,center=0,vmin=-1,vmax=1,ax=ax,cbar_kws={"label":"Pearson correlation"}); ax.set(title="Correlations among numeric columns",xlabel="Feature",ylabel="Feature"); ax.tick_params(axis="x",rotation=EDA_X_LABEL_ROTATION); save(fig,"correlation")

## 12. Structural-break screen

The rolling summaries help distinguish a sustained low period from isolated quiet readings. This is a visual screen, not a formal change-point test; sampling gaps can also affect daily means.

In [ ]:
rm=daily.rolling(EDA_ROLLING_WINDOW,min_periods=EDA_ROLLING_MIN_PERIODS).mean(); rq=daily.rolling(EDA_ROLLING_WINDOW,min_periods=EDA_ROLLING_MIN_PERIODS).quantile(EDA_STRUCTURAL_LOW_QUANTILE); fig,ax=plt.subplots(figsize=EDA_WIDE_FIGURE_SIZE); ax.plot(daily.index,daily,alpha=EDA_DAILY_LINE_ALPHA,label="Daily mean"); ax.plot(rm.index,rm,linewidth=EDA_LINE_WIDTH,label=f"{EDA_ROLLING_WINDOW} mean"); ax.plot(rq.index,rq,linestyle="--",label="Rolling 10th percentile"); ax.set(title="Visual structural-change screen",xlabel="Date (source local)",ylabel="People (count)"); ax.legend(); save(fig,"structural_break"); trough=rm.idxmin(); print(f"lowest rolling mean={rm.min():.2f}, date={trough:%Y-%m-%d}")

## Key findings

1. The count target has zeros and an upper tail. Keep genuine zero readings, use MAE as primary metric, and inspect outliers rather than deleting them automatically.
2. Occupancy has a repeating daily cycle and weekday/weekend differences. Use cyclic hour encoding and a weekend indicator.
3. Day and hour jointly matter. Include a day-hour interaction or a model that learns interactions.
4. Semester flags are associated with different crowd levels. Keep them in Forecast features and validate chronologically.
5. The holiday indicator has sparse support. Retain it cautiously and avoid strong conclusions from it.
6. Temperature is associated with occupancy, and the series has a low period near its end. Combine weather and calendar context; validate on later data.

All patterns are descriptive, not causal. The following cell writes a measured summary to `reports/eda_summary.md`.

In [ ]:
from src.config import EDA_SUMMARY_PATH, REPORTS_DIR
REPORTS_DIR.mkdir(parents=True,exist_ok=True)
wh=df[df.is_weekend==0].groupby("hour")[TARGET_COLUMN].mean(); we=df[df.is_weekend==1].groupby("hour")[TARGET_COLUMN].mean(); sem=df.groupby("is_during_semester")[TARGET_COLUMN].mean(); holiday_dates=df.loc[df.is_holiday==1,"local_date"].nunique()
lines=["# Exploratory Data Analysis Summary","",f"The supplied data contains **{len(df):,} rows** and **{len(raw.columns)} raw columns**, spanning source-local dates **{df.local_datetime.min():%Y-%m-%d}** to **{df.local_datetime.max():%Y-%m-%d}**. No rows were dropped or altered.","","## Key findings and modelling implications","",f"1. **Target distribution:** skew={skew:.3f}; zero readings={zeros:,} ({zeros/len(df):.1%}); above upper 1.5-IQR fence={outliers:,}. Keep zero and high counts; use MAE and investigate large errors.",f"2. **Daily rhythm:** weekday peak={int(wh.idxmax()):02d}:00 ({wh.max():.2f}); weekend peak={int(we.idxmax()):02d}:00 ({we.max():.2f}). Encode hour cyclically and include weekend.",f"3. **Day-hour interaction:** busiest heatmap cell={max_day}, {max_hour:02d}:00, mean={max_value:.2f}. Represent joint calendar effects.",f"4. **Semester association:** during-semester mean={sem.get(1,float('nan')):.2f}; outside-semester mean={sem.get(0,float('nan')):.2f}. Retain calendar flags and validate on later dates.",f"5. **Sparse holiday indicator:** {int((df.is_holiday==1).sum()):,} readings across {holiday_dates} dates. Retain as context but interpret cautiously.",f"6. **Weather and time variation:** temperature Pearson r={r:.3f}; lowest {EDA_ROLLING_WINDOW} rolling daily mean={rm.min():.2f} on {trough:%Y-%m-%d}. Combine weather with calendar context and use chronological validation.","","## Interpretation limits","","These are descriptive associations, not causal claims. The data cover less than two full years, holiday support is sparse, and the structural-break chart is not a formal change-point test. Correlation does not capture cyclical hour effects. Figures are saved under `reports/figures/` at 200 dpi.",""]
EDA_SUMMARY_PATH.write_text("\n".join(lines),encoding="utf-8")
print(f"Wrote {EDA_SUMMARY_PATH}")